In [2]:
import pandas as pd

path = "/storage/vacoda/data/interim/hs_sync/target_h5/applied/comtrade_target_h5_hs_synced.parquet"

df = pd.read_parquet(path)

In [15]:
df.head(3)

,hs_sync_input_row_id,type_code,frequency_code,ref_period_id,year,month,period,reporter_code,reporter_iso,reporter_name,...,source_cmd_original,target_hs,target_cmd,allocation_weight,conversion_need,hs_sync_method,allocation_strategy,allocation_is_assumption,primary_value_original,primary_value_hs_synced
0,1,C,A,20170101,2017,52,2017,28,ATG,Antigua and Barbuda,...,391211,H5,391211,1.0,easy_without_weights,direct_1_to_1,deterministic,False,169.270,169.270
1,2,C,A,20170101,2017,52,2017,28,ATG,Antigua and Barbuda,...,391211,H5,391211,1.0,easy_without_weights,direct_1_to_1,deterministic,False,169.270,169.270
2,3,C,A,20170101,2017,52,2017,28,ATG,Antigua and Barbuda,...,391211,H5,391211,1.0,easy_without_weights,direct_1_to_1,deterministic,False,38.952,38.952


In [4]:
df.columns.tolist()

['hs_sync_input_row_id',
 'type_code',
 'frequency_code',
 'ref_period_id',
 'year',
 'month',
 'period',
 'reporter_code',
 'reporter_iso',
 'reporter_name',
 'flow_code',
 'flow_name',
 'partner_code',
 'partner_iso',
 'partner_name',
 'partner2_code',
 'partner2_iso',
 'partner2_name',
 'source_hs',
 'classification_search',
 'is_original_classification',
 'source_cmd',
 'commodity_name',
 'aggregation_level',
 'is_leaf',
 'customs_code',
 'customs_name',
 'mos_code',
 'mot_code',
 'mot_name',
 'quantity_unit_code',
 'quantity_unit',
 'quantity',
 'is_quantity_estimated',
 'alt_quantity_unit_code',
 'alt_quantity_unit',
 'alt_quantity',
 'is_alt_quantity_estimated',
 'net_weight_kg',
 'is_net_weight_estimated',
 'gross_weight_kg',
 'is_gross_weight_estimated',
 'cif_value_usd',
 'fob_value_usd',
 'primary_value',
 'legacy_estimation_flag',
 'is_reported',
 'is_aggregate',
 'source_hs_original',
 'source_cmd_original',
 'target_hs',
 'target_cmd',
 'allocation_weight',
 'conversion_n

In [6]:
value_columns =['cif_value_usd',
 'fob_value_usd',
 'primary_value']

df[value_columns].describe()

,cif_value_usd,fob_value_usd,primary_value
count,3.345576e+07,3.344003e+07,4.991792e+07
mean,8.056602e+05,1.088909e+06,1.187314e+06
std,1.709283e+07,2.401075e+07,2.327669e+07
min,0.000000e+00,0.000000e+00,0.000000e+00
25%,7.119700e+01,1.159340e+02,4.822450e+02
50%,1.992000e+03,3.340467e+03,6.148100e+03
75%,3.630400e+04,5.159355e+04,7.244236e+04
max,1.139287e+10,1.298835e+10,1.298835e+10


In [7]:
value_cols = [
    "primary_value",
    "cif_value_usd",
    "fob_value_usd",
]

df[value_cols].isna().value_counts().sort_index()

primary_value  cif_value_usd  fob_value_usd
False          False          False            16977876
                              True             16477885
               True           False            16462156
Name: count, dtype: int64

In [8]:
pd.DataFrame({
    "rows": [
        len(df),
        df["primary_value"].isna().sum(),
        (
            df["primary_value"].isna()
            & df["cif_value_usd"].notna()
        ).sum(),
        (
            df["primary_value"].isna()
            & df["fob_value_usd"].notna()
        ).sum(),
        (
            df["primary_value"].isna()
            & (
                df["cif_value_usd"].notna()
                | df["fob_value_usd"].notna()
            )
        ).sum(),
    ]
}, index=[
    "total_rows",
    "primary_missing",
    "primary_missing_but_cif_present",
    "primary_missing_but_fob_present",
    "primary_missing_but_any_alternative_present",
])

,rows
total_rows,49917917
primary_missing,0
primary_missing_but_cif_present,0
primary_missing_but_fob_present,0
primary_missing_but_any_alternative_present,0


In [9]:
df[value_cols].eq(0).sum()

primary_value         44
cif_value_usd    5202168
fob_value_usd    4077697
dtype: int64

In [10]:
availability = (
    df[
        [
            "primary_value",
            "cif_value_usd",
            "fob_value_usd",
        ]
    ]
    .notna()
    .value_counts()
    .rename("rows")
    .reset_index()
)

availability

,primary_value,cif_value_usd,fob_value_usd,rows
0,True,True,True,16977876
1,True,True,False,16477885
2,True,False,True,16462156


In [11]:
comparison = pd.DataFrame({
    "primary_equals_cif": (
        df["primary_value"].eq(df["cif_value_usd"])
        & df["cif_value_usd"].notna()
    ),
    "primary_equals_fob": (
        df["primary_value"].eq(df["fob_value_usd"])
        & df["fob_value_usd"].notna()
    ),
})

comparison.value_counts().rename("rows")

primary_equals_cif  primary_equals_fob
True                False                 26510456
False               True                  21732470
True                True                   1674991
Name: rows, dtype: int64

In [12]:
pd.Series({
    "primary equals CIF": (
        df["primary_value"].eq(df["cif_value_usd"])
        & df["cif_value_usd"].notna()
    ).sum(),

    "primary equals FOB": (
        df["primary_value"].eq(df["fob_value_usd"])
        & df["fob_value_usd"].notna()
    ).sum(),

    "primary equals both": (
        df["primary_value"].eq(df["cif_value_usd"])
        & df["primary_value"].eq(df["fob_value_usd"])
        & df["cif_value_usd"].notna()
        & df["fob_value_usd"].notna()
    ).sum(),

    "primary differs from both": (
        df["cif_value_usd"].notna()
        & df["fob_value_usd"].notna()
        & df["primary_value"].ne(df["cif_value_usd"])
        & df["primary_value"].ne(df["fob_value_usd"])
    ).sum(),
})


primary equals CIF           28185447
primary equals FOB           23407461
primary equals both           1674991
primary differs from both           0
dtype: int64

In [13]:
df.assign(
    primary_equals_cif=(
        df["primary_value"].eq(df["cif_value_usd"])
        & df["cif_value_usd"].notna()
    ),
    primary_equals_fob=(
        df["primary_value"].eq(df["fob_value_usd"])
        & df["fob_value_usd"].notna()
    ),
).groupby("flow_code").agg(
    rows=("primary_value", "size"),
    cif_available=("cif_value_usd", "count"),
    fob_available=("fob_value_usd", "count"),
    primary_equals_cif=("primary_equals_cif", "sum"),
    primary_equals_fob=("primary_equals_fob", "sum"),
)

,rows,cif_available,fob_available,primary_equals_cif,primary_equals_fob
flow_code,,,,,
DX,4485514,1100496,4485514,3478,4485514
FM,4465801,4275553,1815395,4095596,563843
M,24938179,23914484,11396053,23573211,2766223
MIP,126993,126993,71503,126993,6108
MOP,28759,28759,8960,28759,440
RM,373089,341063,163033,333977,85759
RX,409186,156055,409186,3924,409186
X,14952413,3442777,14952405,19297,14952405
XIP,34433,9468,34433,0,34433


In [16]:
keys = [
    "period",
    "reporter_code",
    "partner_code",
    "target_cmd",
]

export_check = (
    df[df["flow_code"].isin(["X", "DX", "RX"])]
    .groupby(keys + ["flow_code"], as_index=False)["primary_value"]
    .sum()
    .pivot_table(
        index=keys,
        columns="flow_code",
        values="primary_value",
        fill_value=0,
    )
    .reset_index()
)

export_check["dx_plus_rx"] = (
    export_check.get("DX", 0)
    + export_check.get("RX", 0)
)

export_check["difference"] = (
    export_check.get("X", 0)
    - export_check["dx_plus_rx"]
)

export_check["relative_difference"] = (
    export_check["difference"].abs()
    / export_check.get("X", 0).replace(0, pd.NA)
)

export_check["relative_difference"].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
)

count     3867229.0
unique      49075.0
top             1.0
freq      2859467.0
Name: relative_difference, dtype: float64

In [17]:
pd.Series({
    "comparable_groups": export_check["relative_difference"].notna().sum(),
    "within_1_percent": export_check["relative_difference"].le(0.01).sum(),
    "within_5_percent": export_check["relative_difference"].le(0.05).sum(),
    "within_10_percent": export_check["relative_difference"].le(0.10).sum(),
})

comparable_groups    3867229
within_1_percent     1002841
within_5_percent     1003356
within_10_percent    1003671
dtype: int64

In [18]:
import numpy as np

x_value = export_check.get("X", 0)

export_check["relative_difference"] = np.where(
    x_value > 0,
    export_check["difference"].abs() / x_value,
    np.nan,
)

export_check["relative_difference"].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
)

count    3.867229e+06
mean     7.401726e-01
std      4.383936e-01
min      0.000000e+00
50%      1.000000e+00
75%      1.000000e+00
90%      1.000000e+00
95%      1.000000e+00
99%      1.000000e+00
max      1.000000e+00
Name: relative_difference, dtype: float64

In [19]:

comparable = export_check.loc[
    export_check.get("X", 0).gt(0)
    & (
        export_check.get("DX", 0).gt(0)
        | export_check.get("RX", 0).gt(0)
    )
].copy()

comparable["dx_plus_rx"] = (
    comparable.get("DX", 0)
    + comparable.get("RX", 0)
)

comparable["relative_difference"] = (
    (
        comparable["X"]
        - comparable["dx_plus_rx"]
    ).abs()
    / comparable["X"]
)

pd.Series({
    "comparable_groups": len(comparable),
    "within_1_percent": comparable["relative_difference"].le(0.01).sum(),
    "within_5_percent": comparable["relative_difference"].le(0.05).sum(),
    "within_10_percent": comparable["relative_difference"].le(0.10).sum(),
})

comparable_groups    1007762
within_1_percent     1002841
within_5_percent     1003356
within_10_percent    1003671
dtype: int64

In [20]:
pd.Series({
    "within_1_percent_pct":
        comparable["relative_difference"].le(0.01).mean() * 100,

    "within_5_percent_pct":
        comparable["relative_difference"].le(0.05).mean() * 100,

    "within_10_percent_pct":
        comparable["relative_difference"].le(0.10).mean() * 100,
})

within_1_percent_pct     99.511690
within_5_percent_pct     99.562794
within_10_percent_pct    99.594051
dtype: float64

In [21]:
export_check["has_x"] = export_check.get("X", 0).gt(0)
export_check["has_dx"] = export_check.get("DX", 0).gt(0)
export_check["has_rx"] = export_check.get("RX", 0).gt(0)

export_check["has_detailed_export"] = (
    export_check["has_dx"]
    | export_check["has_rx"]
)

pd.Series({
    "groups_with_x": export_check["has_x"].sum(),

    "groups_with_detailed_export":
        export_check["has_detailed_export"].sum(),

    "detailed_groups_without_x": (
        ~export_check["has_x"]
        & export_check["has_detailed_export"]
    ).sum(),

    "dx_without_x": (
        ~export_check["has_x"]
        & export_check["has_dx"]
    ).sum(),

    "rx_without_x": (
        ~export_check["has_x"]
        & export_check["has_rx"]
    ).sum(),
})

groups_with_x                  3867229
groups_with_detailed_export    1007762
detailed_groups_without_x            0
dx_without_x                         0
rx_without_x                         0
dtype: int64

In [22]:
export_fallback = export_check.loc[
    ~export_check["has_x"]
    & export_check["has_detailed_export"]
].copy()

pd.Series({
    "fallback_dx_value":
        export_fallback.get("DX", 0).sum(),

    "fallback_rx_value":
        export_fallback.get("RX", 0).sum(),

    "fallback_total_export_value":
        (
            export_fallback.get("DX", 0)
            + export_fallback.get("RX", 0)
        ).sum(),
})

fallback_dx_value              0.0
fallback_rx_value              0.0
fallback_total_export_value    0.0
dtype: float64

In [24]:
## 
value_columns = ['primary_value', 'primary_value_hs_synced', 'primary_value_original']
df[value_columns].describe()

,primary_value,primary_value_hs_synced,primary_value_original
count,4.991792e+07,4.991792e+07,4.991792e+07
mean,1.187314e+06,1.163947e+06,1.187314e+06
std,2.327669e+07,2.191921e+07,2.327669e+07
min,0.000000e+00,0.000000e+00,0.000000e+00
25%,4.822450e+02,4.736500e+02,4.822450e+02
50%,6.148100e+03,6.033479e+03,6.148100e+03
75%,7.244236e+04,7.107164e+04,7.244236e+04
max,1.298835e+10,1.298835e+10,1.298835e+10


In [25]:
import duckdb

path = (
    "/storage/vacoda/data/interim/cif_fob/"
    "target_h5/target_h5_cif_fob_mirror_pairs.parquet"
)

df = duckdb.sql(
    f"""
    SELECT *
    FROM read_parquet('{path}')
    LIMIT 100
    """
).df()

df.head()

,year,exporter_iso,importer_iso,target_cmd,export_value_fob,import_value_cif,export_net_weight_kg,import_net_weight_kg,export_direct_fob_value,export_direct_cif_value,...,has_export_report,has_import_report,has_both_mirrors,raw_value_ratio,export_unit_value_usd_per_kg,import_unit_value_usd_per_kg,unit_value_ratio,quantity_concordance,exclusion_reason,training_eligible
0,2023,USA,NLD,440399,1.237298e+06,4.291339e+04,4.608168e+06,27263.0,1.237298e+06,NaN,...,True,True,True,0.034683,0.268501,1.574052,5.862370,0.005916,low_quantity_concordance,False
1,2023,USA,THA,440399,6.409000e+04,6.794644e+03,2.386955e+05,8834.0,6.409000e+04,NaN,...,True,True,True,0.106017,0.268501,0.769147,2.864596,0.037009,low_quantity_concordance,False
2,2024,BEL,PRT,440399,2.460930e+02,4.825988e+04,7.352000e+01,69513.0,2.460930e+02,NaN,...,True,True,True,196.104249,3.347293,0.694257,0.207408,0.001058,low_quantity_concordance,False
3,2024,CAN,CHN,440399,1.065437e+07,1.403722e+07,4.277231e+07,22764763.0,1.065437e+07,NaN,...,True,True,True,1.317507,0.249095,0.616620,2.475441,0.532231,low_quantity_concordance,False
4,2024,HRV,BIH,440399,8.581400e+04,1.365331e+05,4.841800e+05,1090633.0,8.581400e+04,NaN,...,True,True,True,1.591035,0.177236,0.125187,0.706330,0.443944,low_quantity_concordance,False


In [26]:
duckdb.sql(
    f"""
    SELECT
        exclusion_reason,
        COUNT(*) AS n_rows,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS share_percent
    FROM read_parquet('{path}')
    GROUP BY exclusion_reason
    ORDER BY n_rows DESC
    """
).df()

,exclusion_reason,n_rows,share_percent
0,low_quantity_concordance,1972869,35.95
1,missing_export_report,1831489,33.37
2,missing_import_report,1328738,24.21
3,eligible,303681,5.53
4,missing_or_nonpositive_import_weight,29993,0.55
5,missing_or_nonpositive_export_weight,21210,0.39
6,nonpositive_export_value,1,0.00


In [27]:
duckdb.sql(
    f"""
    SELECT
        QUANTILE_CONT(
            quantity_concordance,
            [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
        ) AS concordance_quantiles
    FROM read_parquet('{path}')
    WHERE quantity_concordance IS NOT NULL
    """
).df()

,concordance_quantiles
0,"[0.0001696709527056308, 0.0034566062186056684,..."


In [29]:
duckdb.sql(
    f"""
    SELECT
        COUNT(*) AS n_eligible,
        MEDIAN(raw_value_ratio) AS median_value_ratio,
        MEDIAN(unit_value_ratio) AS median_unit_value_ratio,
        QUANTILE_CONT(
            unit_value_ratio,
            [0.01, 0.05, 0.50, 0.95, 0.99]
        ) AS unit_value_ratio_quantiles
    FROM read_parquet('{path}')
    WHERE training_eligible
    """
).df()

,n_eligible,median_value_ratio,median_unit_value_ratio,unit_value_ratio_quantiles
0,303681,1.047962,1.048389,"[0.1622232548794118, 0.5279949415938775, 1.048..."


In [30]:
import duckdb

path = (
    "/storage/vacoda/data/interim/cif_fob/"
    "target_h5/target_h5_cif_fob_mirror_pairs.parquet"
)

duckdb.sql(
    f"""
    SELECT
        COUNT(*) AS n_eligible,

        COUNT(*) FILTER (
            WHERE unit_value_ratio < 1
        ) AS n_ratio_below_1,

        COUNT(*) FILTER (
            WHERE unit_value_ratio BETWEEN 1 AND 1.5
        ) AS n_ratio_1_to_1_5,

        COUNT(*) FILTER (
            WHERE unit_value_ratio > 1.5
        ) AS n_ratio_above_1_5,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE unit_value_ratio < 1
            ) / COUNT(*),
            2
        ) AS share_below_1_percent

    FROM read_parquet('{path}')
    WHERE training_eligible
    """
).df()

,n_eligible,n_ratio_below_1,n_ratio_1_to_1_5,n_ratio_above_1_5,share_below_1_percent
0,303681,97331,174653,31697,32.05


In [33]:
import duckdb

path_hs_sync = (
    "/storage/vacoda/data/interim/hs_sync/target_h5/applied/"
    "comtrade_target_h5_hs_synced.parquet"
)

duckdb.sql(
    f"""
    SELECT
        is_quantity_estimated,
        is_net_weight_estimated,
        COUNT(*) AS n_rows,

        MEDIAN(quantity / net_weight_kg) FILTER (
            WHERE quantity > 0
              AND net_weight_kg > 0
              AND LOWER(quantity_unit) = 'kg'
        ) AS median_quantity_to_net_weight

    FROM read_parquet('{path_hs_sync}')

    GROUP BY
        is_quantity_estimated,
        is_net_weight_estimated

    ORDER BY
        is_quantity_estimated,
        is_net_weight_estimated
    """
).df()

,is_quantity_estimated,is_net_weight_estimated,n_rows,median_quantity_to_net_weight
0,False,False,41037308,1.000000
1,False,True,557736,0.999173
2,True,False,2800718,1.161457
3,True,True,5522155,1.000000


In [34]:
duckdb.sql(
    f"""
    SELECT
        is_quantity_estimated,
        is_net_weight_estimated,

        COUNT(*) FILTER (
            WHERE quantity > 0
              AND net_weight_kg > 0
              AND LOWER(TRIM(quantity_unit)) = 'kg'
        ) AS n_comparable_kg_rows,

        MEDIAN(quantity / net_weight_kg) FILTER (
            WHERE quantity > 0
              AND net_weight_kg > 0
              AND LOWER(TRIM(quantity_unit)) = 'kg'
        ) AS median_quantity_to_net_weight,

        QUANTILE_CONT(
            quantity / net_weight_kg,
            [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
        ) FILTER (
            WHERE quantity > 0
              AND net_weight_kg > 0
              AND LOWER(TRIM(quantity_unit)) = 'kg'
        ) AS ratio_quantiles

    FROM read_parquet('{path_hs_sync}')

    GROUP BY
        is_quantity_estimated,
        is_net_weight_estimated

    ORDER BY
        is_quantity_estimated,
        is_net_weight_estimated
    """
).df()

,is_quantity_estimated,is_net_weight_estimated,n_comparable_kg_rows,median_quantity_to_net_weight,ratio_quantiles
0,False,False,37547512,1.000000,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]"
1,False,True,2,0.999173,"[0.9991730843553875, 0.9991730843553874, 0.999..."
2,True,False,570,1.161457,"[0.011277777777777779, 0.3178672421601963, 0.7..."
3,True,True,5035355,1.000000,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]"
